# Yad2 + Madlan Quick Demo

This notebook runs a **quick, demo-sized** test of both **Yad2** and **Madlan** pipelines with **all main filters and CLI-style args defined in the cells below**. No need to edit `scraper_preferences.json` or the CLI.

- **Demo size:** max 4 search pages per pipeline so the run finishes in a few minutes.
- **Full verbosity:** logging at INFO and pretty-printed run plans before each scrape.
- **Unified locations:** use English names (e.g. `Haifa`, `Rehovot`); they resolve to the correct areas/locations for each platform.

Edit the **Config** cell to set locations, date cutoff (−1 month), max building floors, cities to avoid, price range, and other filters, then run all cells.

**Jupyter async fix:** Run this cell first so that async cells (e.g. Playwright with `await`) work. It allows nested event loops and avoids "This event loop is already running" errors.

In [1]:
import nest_asyncio
nest_asyncio.apply()
print("nest_asyncio applied — async/await cells will run correctly.")

nest_asyncio applied — async/await cells will run correctly.


## 1. User-defined config (all test parameters)

Set every filter and input the pipelines support. These values are used for both Yad2 and Madlan where applicable.

In [2]:
# ========== CONFIG: edit these for your demo run ==========
from pathlib import Path

CONFIG = {
    # Unified location(s) — same for Yad2 and Madlan (English or legacy names, comma-separated)
    "locations": "Haifa_Area",
    # Date filter: only listings from the last N months (1 = last month from today)
    "publication_max_months": 1,
    # Max total floors in building (listings in taller buildings are skipped)
    "max_building_floors": 4,
    # Cities to exclude (post-filter)
    "exclude_cities": ["Kiryat Yam"],
    # Neighborhoods to exclude (Madlan; Yad2 uses only exclude_cities)
    "exclude_neighborhoods": [],
    # Price range (ILS)
    "price_min": 1_900_000,
    "price_max": 2_500_000,
    # Rooms (Yad2 uses URL filters; Madlan has rooms_min/rooms_max)
    "rooms_min": 4,
    "rooms_max": 6,
    # Max floor of apartment, min sqm
    "max_floor": 4,
    "min_square_meters": 90,
    # Property condition: Yad2 uses [5, 3]; Madlan uses ["toRenovated", "preserved"]
    "property_condition_yad2": [5, 3],
    "property_condition_madlan": ["toRenovated", "preserved"],
    # Private-only (Yad2: skip broker cards/listings); Madlan: same idea = private_only_madlan
    "private_only": False,
    "private_only_madlan": False,
    # Demo run: max 4 search pages per pipeline
    "max_pages": 4,
    "headless": False,
    "captcha_avoidance_min": 0.0,
    # Output dirs (separate for Yad2 and Madlan)
    "output_dir_yad2": Path("output_demo_yad2"),
    "output_dir_madlan": Path("output_demo_madlan"),
}

### Effective config summary

Print the config that will be used so you can verify date cutoff, exclude cities, max floors, and locations before running.

In [3]:
from datetime import datetime, timedelta

cutoff_days = CONFIG["publication_max_months"] * 30
cutoff_date = (datetime.now() - timedelta(days=cutoff_days)).strftime("%Y-%m-%d")
print("Effective config (demo)")
print("  locations (unified):     ", repr(CONFIG["locations"]))
print("  publication cutoff:      last", CONFIG["publication_max_months"], "month(s) → drop listings before", cutoff_date)
print("  max_building_floors:     ", CONFIG["max_building_floors"])
print("  exclude_cities:          ", CONFIG["exclude_cities"] or "(none)")
print("  exclude_neighborhoods:   ", CONFIG["exclude_neighborhoods"] or "(none)")
print("  price_min – price_max:   ", CONFIG["price_min"], "–", CONFIG["price_max"])
print("  rooms_min – rooms_max:   ", CONFIG["rooms_min"], "–", CONFIG["rooms_max"])
print("  max_floor, min_sqm:      ", CONFIG["max_floor"], ",", CONFIG["min_square_meters"])
print("  private_only (Yad2):     ", CONFIG["private_only"])
print("  private_only_madlan (Madlan):", CONFIG["private_only_madlan"])
print("  max_pages (per pipeline):", CONFIG["max_pages"])
print("  headless:                ", CONFIG["headless"])
print("  output_dir_yad2:         ", CONFIG["output_dir_yad2"])
print("  output_dir_madlan:       ", CONFIG["output_dir_madlan"])

Effective config (demo)
  locations (unified):      'Haifa_Area'
  publication cutoff:      last 1 month(s) → drop listings before 2026-02-13
  max_building_floors:      4
  exclude_cities:           ['Kiryat Yam']
  exclude_neighborhoods:    (none)
  price_min – price_max:    1900000 – 2500000
  rooms_min – rooms_max:    4 – 6
  max_floor, min_sqm:       4 , 90
  private_only (Yad2):      False
  seller_type (Madlan):     all
  max_pages (per pipeline): 4
  headless:                 False
  output_dir_yad2:          output_demo_yad2
  output_dir_madlan:        output_demo_madlan


---
## 2. Yad2 pipeline

Steps: resolve `--locations` to Yad2 areas/cities and export slug, build the filter-preferences dict (same shape as `scraper_preferences.json`), then run the full Yad2 scraper with full verbosity and pretty-printed run plan.

### 2.1 Resolve locations to Yad2 areas/cities

Unified input (e.g. `"Haifa"`) is resolved via `assets/unified_location_names.json` to Yad2 area names (e.g. `["Haifa Area"]`) and optional cities. The same string also yields an **export_slug** used for the Excel filename (e.g. `Haifa_Area.xlsx`).

In [4]:
from unified_locations import resolve_locations_to_yad2

locations_str = CONFIG["locations"].strip()
yad2_areas, yad2_cities, export_slug = resolve_locations_to_yad2(locations_str)
print("Resolved for Yad2:")
print("  areas:      ", yad2_areas or "(none, will use preferences/default)")
print("  cities:     ", yad2_cities or "(none)")
print("  export_slug:", export_slug)

Unknown location 'Haifa_Area'; add to assets/unified_location_names.json


Resolved for Yad2:
  areas:       (none, will use preferences/default)
  cities:      (none)
  export_slug: listings


### 2.2 Build Yad2 filter preferences dict

Build the same structure that `yad2_pipeline.load_filter_preferences()` returns: `url_filters` (price, max_floor, min_sqm, property_condition), `post_filters` (publication_cutoff_months, max_floor_total, cities_to_skip, private_only), plus `areas`, `cities`, `district`, `listing_type`.

In [ ]:
# Same shape as scraper_preferences / filter_preferences (nested)
filter_preferences_yad2 = {
    "district": "Center and Sharon",
    "listing_type": "forsale",
    "areas": yad2_areas or [],
    "cities": yad2_cities or [],
    "url_filters": {
        "minPrice": CONFIG["price_min"],
        "maxPrice": CONFIG["price_max"],
        "maxFloor": CONFIG["max_floor"],
        "minSquareMeterBuild": CONFIG["min_square_meters"],
        "propertyCondition": CONFIG["property_condition_yad2"],
    },
    "post_filters": {
        "publication_cutoff_months": CONFIG["publication_max_months"],
        "max_floor_total": CONFIG["max_building_floors"],
        "cities_to_skip": list(CONFIG["exclude_cities"]),
        "private_only": CONFIG["private_only"],
    },
}
print("Filter preferences (Yad2) built. Keys:", list(filter_preferences_yad2.keys()))

### 2.3 Run Yad2 scraper (full verbosity)

Set logging to **INFO** so every step is printed. Instantiate `Yad2Scraper` with the filter preferences, resolved areas/cities, export slug, and demo limits (max_pages=4). Calling `scraper.scrape()` will first **pretty-print the run plan** (areas, cities, filters, exclude cities, date cutoff, max floors), then run the full pipeline.

In [ ]:
import logging
import os
from dotenv import load_dotenv
from yad2_pipeline import Geocoder, RouteCalculator, Yad2Scraper

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)

email = os.getenv("GEOCODING_EMAIL", "example@example.com")
geocoder_yad2 = Geocoder(email=email)
ors_key = os.getenv("ORS_API_KEY")
route_calc_yad2 = RouteCalculator(api_key=ors_key) if ors_key else None

scraper_yad2 = Yad2Scraper(
    output_dir=CONFIG["output_dir_yad2"],
    geocoder=geocoder_yad2,
    route_calculator=route_calc_yad2,
    max_pages=CONFIG["max_pages"],
    captcha_avoidance_min=CONFIG["captcha_avoidance_min"],
    headless=CONFIG["headless"],
    cities_to_skip=list(CONFIG["exclude_cities"]),
    areas=yad2_areas if yad2_areas else None,
    cities=yad2_cities if yad2_cities else None,
    filter_preferences=filter_preferences_yad2,
    export_slug=export_slug or None,
)
print("Running Yad2 full pipeline (run plan will appear next)...")
scraper_yad2.scrape()
print("Yad2 scrape done. CSV:", scraper_yad2.output_csv)

### 2.4 Inspect Yad2 output

Read the CSV and run summary. List the generated Excel file (named from `export_slug`, e.g. `Haifa_Area.xlsx`).

In [ ]:
import json
import pandas as pd

out_yad2 = CONFIG["output_dir_yad2"]
if (out_yad2 / "run_summary.json").exists():
    summary = json.loads((out_yad2 / "run_summary.json").read_text(encoding="utf-8"))
    print("Yad2 run_summary.json:")
    for k, v in summary.items():
        print(f"  {k}: {v}")
if (out_yad2 / "listings_full.csv").exists():
    df_yad2 = pd.read_csv(out_yad2 / "listings_full.csv")
    print("\nListings CSV shape:", df_yad2.shape)
    print(df_yad2.head() if len(df_yad2) > 0 else "(no rows)")
excel_name = f"{export_slug}.xlsx" if export_slug else "fixed_hebrew_file.xlsx"
excel_path = out_yad2 / excel_name
print("\nExcel output:", excel_path, "(exists:" if excel_path.exists() else "(not found)")

---
## 3. Madlan pipeline

Same idea: resolve `--locations` to Madlan Hebrew locations and export slug, build the Madlan preferences dict, then run the full Madlan scraper with full verbosity and pretty-printed run plan (locations, filters, exclude cities/neighborhoods, date cutoff, max floors).

### 3.1 Resolve locations to Madlan (Hebrew list)

The same unified input (e.g. `"Haifa"`) is resolved to Madlan’s Hebrew location list (e.g. `["חיפה"]`) and the same **export_slug** for the Excel filename.

In [ ]:
from unified_locations import resolve_locations_to_madlan

madlan_locations, madlan_export_slug = resolve_locations_to_madlan(locations_str)
print("Resolved for Madlan:")
print("  locations (Hebrew):", madlan_locations or "(none)")
print("  export_slug:       ", madlan_export_slug)

### 3.2 Build Madlan preferences dict

Build the same structure as `madlan_pipeline.load_madlan_preferences()`: locations, price_min/max, rooms_min/max, publication_max_months, max_building_floors, exclude_cities, exclude_neighborhoods, private_only_madlan, etc.

In [ ]:
madlan_prefs = {
    "locations": madlan_locations if madlan_locations else ["חיפה"],
    "price_min": CONFIG["price_min"],
    "price_max": CONFIG["price_max"],
    "rooms_min": CONFIG["rooms_min"],
    "rooms_max": CONFIG["rooms_max"],
    "property_condition": CONFIG["property_condition_madlan"],
    "private_only_madlan": CONFIG["private_only_madlan"],
    "max_floor": CONFIG["max_floor"],
    "min_square_meters": CONFIG["min_square_meters"],
    "publication_max_months": CONFIG["publication_max_months"],
    "max_building_floors": CONFIG["max_building_floors"],
    "exclude_cities": list(CONFIG["exclude_cities"]),
    "exclude_neighborhoods": list(CONFIG["exclude_neighborhoods"]),
    "private_only": CONFIG["private_only"],
    "trust_url_seller_filter": True,
    "captcha_avoidance_min": CONFIG["captcha_avoidance_min"],
}
print("Madlan preferences built. Locations:", madlan_prefs["locations"])

### 3.3 Run Madlan scraper (full verbosity)

Set logging to INFO (if not already). Instantiate `MadlanScraper` with the preferences and export slug. Calling `scraper.scrape()` will **pretty-print the run plan** (locations, price, rooms, exclude cities/neighborhoods, publication cutoff, max building floors), then run the full pipeline.

In [ ]:
from madlan_pipeline import MadlanScraper

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)

scraper_madlan = MadlanScraper(
    output_dir=CONFIG["output_dir_madlan"],
    geocoder=geocoder_yad2,
    route_calculator=route_calc_yad2,
    max_pages=CONFIG["max_pages"],
    headless=CONFIG["headless"],
    madlan_preferences=madlan_prefs,
    export_slug=madlan_export_slug or None,
)
print("Running Madlan full pipeline (run plan will appear next)...")
scraper_madlan.scrape()
print("Madlan scrape done. CSV:", scraper_madlan.output_csv)

### 3.4 Inspect Madlan output

Read Madlan CSV and run summary. List the generated Excel (same naming: export_slug.xlsx).

In [ ]:
out_madlan = CONFIG["output_dir_madlan"]
if (out_madlan / "run_summary.json").exists():
    summary_m = json.loads((out_madlan / "run_summary.json").read_text(encoding="utf-8"))
    print("Madlan run_summary.json:")
    for k, v in summary_m.items():
        print(f"  {k}: {v}")
if (out_madlan / "listings_full.csv").exists():
    df_madlan = pd.read_csv(out_madlan / "listings_full.csv")
    print("\nListings CSV shape:", df_madlan.shape)
    print(df_madlan.head() if len(df_madlan) > 0 else "(no rows)")
excel_m = f"{madlan_export_slug}.xlsx" if madlan_export_slug else "fixed_hebrew_file.xlsx"
excel_madlan_path = out_madlan / excel_m
print("\nExcel output:", excel_madlan_path, "(exists)" if excel_madlan_path.exists() else "(not found)")

---
**Done.** Both pipelines ran with the config above. CSV and formatted Excel (column order, date/phone/number formats) are in the output dirs. Change the **Config** cell and re-run from there to test other locations or filters (e.g. `publication_max_months: 1`, `exclude_cities`, `max_building_floors`).

# One-page Yad2 scraping demo

This section demonstrates running the new pipeline on **a single search page** and **one listing**:

1. Load page 1 of the Center & Sharon sale listings with the required filters.
2. Extract the first listing and parse its core fields.
3. Inspect parsed fields to verify correctness.
4. Enrich the same listing with geocoding and driving distances (road transportation) to Tel Aviv Savidor and Beer Sheva Center.

You can run just the cells in this section without running the full pipeline.

In [6]:
from pathlib import Path
import os
from pprint import pprint
from datetime import datetime

from dotenv import load_dotenv

from yad2_pipeline import (
    Geocoder,
    RouteCalculator,
    Yad2Scraper,
)
from yad2_url_builder import load_mappings, build_yad2_url_from_json

load_dotenv()

# For the demo we force headful so you can see and solve captcha pages
HEADLESS = False  # we explicitly override headless to False for interactive demo
GEOCODING_EMAIL = os.getenv("GEOCODING_EMAIL", "example@example.com")
ORS_API_KEY = os.getenv("ORS_API_KEY")

# Browser configuration for the demo:
#   "firefox"  -> use Firefox
#   "chromium" -> use Chromium/Chrome
BROWSER_ENGINE = "chromium"  # change to "firefox" if you prefer

output_demo_dir = Path("output_demo")
output_demo_dir.mkdir(exist_ok=True)

# Load canonical Yad2 mappings used by the main pipeline
YAD2_MAPPINGS = load_mappings(Path("assets") / "yad2_area_IDs.json")

geocoder = Geocoder(email=GEOCODING_EMAIL)
route_calculator = RouteCalculator(api_key=ORS_API_KEY) if ORS_API_KEY else None

scraper = Yad2Scraper(output_dir=output_demo_dir, geocoder=geocoder, route_calculator=route_calculator)


def ts(msg: str) -> None:
    """Simple timestamped progress print for the demo cells."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")


print(f"HEADLESS={HEADLESS}, BROWSER_ENGINE={BROWSER_ENGINE}, demo output dir={output_demo_dir.resolve()}")

HEADLESS=False, BROWSER_ENGINE=chromium, demo output dir=/Users/nircko/GIT/web_agent/output_demo


In [7]:
def build_filtered_url(page_number: int = 1, area_name: str | None = None) -> str:
    """Use the same filters as the main pipeline but for a single page.

    This demo now uses the same mapping-driven URL builder as the main
    pipeline. You can optionally pass an `area_name` that matches one of
    the keys under the `"area"` section in `assets/yad2_area_IDs.json`,
    e.g. "Rishon LeZion Area" or "Netanya Area".
    """

    areas_param = [area_name] if area_name else None

    return build_yad2_url_from_json(
        YAD2_MAPPINGS,
        big_area="Center and Sharon",
        listing_type="forsale",
        areas=areas_param,
        cities=None,
        neighborhoods=None,
        min_price=1600000,
        max_price=3600000,
        max_floor=4,
        min_square_meter_build=90,
        property_condition=[5, 3],
        page=page_number,
    )


def extract_listing_id_from_url(url: str) -> str:
    """Small helper mirroring the logic in `Yad2Scraper` for IDs."""
    import re

    base = url.split("?")[0]
    m = re.search(r"/(\d+)$", base)
    if m:
        return m.group(1)
    m = re.search(r"itemId=(\d+)", url)
    if m:
        return m.group(1)
    # Fallback: stable hash
    return f"urlhash_{abs(hash(url))}"


# Choose a demo area that exists in assets/yad2_area_IDs.json under "area".
DEMO_AREA = "Rishon LeZion Area"
search_url_demo = build_filtered_url(page_number=1, area_name=DEMO_AREA)
print("Demo search URL:", search_url_demo)
print("Demo area:", DEMO_AREA)

Demo search URL: https://www.yad2.co.il/realestate/forsale/center-and-sharon?minPrice=1600000&maxPrice=3600000&maxFloor=4&minSquareMeterBuild=90&propertyCondition=5%2C3&page=1


In [8]:
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

first_listing_id = None
first_listing_url = None

async def load_first_listing():
    """Open search page 1, using the configured browser engine, and grab the first listing URL.

    BROWSER_ENGINE is set in the previous cell ("firefox" or "chromium").
    """
    global first_listing_id, first_listing_url

    ts(f"Starting load_first_listing with engine={BROWSER_ENGINE}")

    async with async_playwright() as p:
        # Choose engine based on config
        if BROWSER_ENGINE == "chromium":
            ts("Launching Chromium (incognito context)")
            browser = await p.chromium.launch(headless=HEADLESS)
        else:
            ts("Launching Firefox (incognito context)")
            browser = await p.firefox.launch(headless=HEADLESS)

        context = await browser.new_context()
        page = await context.new_page()

        ts(f"Navigating to search URL: {search_url_demo}")
        await page.goto(
            search_url_demo,
            wait_until="domcontentloaded",
            timeout=120_000,
        )

        # Make sure body is ready
        await page.wait_for_selector("body", timeout=30_000)

        # Try to close cookie / consent popup if present (best-effort)
        try:
            ts("Trying to close consent / cookie popup (if any)")
            btn = await page.wait_for_selector("button:has-text('מאשר')", timeout=5000)
            await btn.click()
            ts("Clicked consent button")
        except PlaywrightTimeoutError:
            ts("No consent button found (timeout), continuing")
        except Exception as e:
            ts(f"Consent button click failed: {e}")

        # If we hit ShieldSquare Captcha, let the user solve it in the visible browser
        while True:
            try:
                title = await page.title()
            except Exception:
                title = ""
            if "ShieldSquare" in title or "Captcha" in title:
                ts("Detected ShieldSquare / Captcha page.")
                ts("Go to the browser window, solve the captcha / challenge until you see the real listings page.")
                input("When the listings are visible in the browser, press Enter here to continue...")
                await page.wait_for_timeout(2000)
                continue  # re-check title
            break  # no longer on captcha page, continue flow

        # Scroll to force lazy-loaded cards
        ts("Scrolling to load listings")
        await page.mouse.wheel(0, 2000)
        await page.wait_for_timeout(2000)

        selector = "a[href*='/realestate/item'], a[href*='/item/']"
        cards = await page.query_selector_all(selector)
        ts(f"Found {len(cards)} potential listing cards on page 1")

        if not cards:
            snippet = (await page.content())[:2000]
            print("\n--- HTML snippet for debugging (first 2000 chars) ---\n")
            print(snippet)
            await browser.close()
            raise RuntimeError("No listing cards found on the demo page (after query_selector_all)")

        first_card = cards[0]
        href = await first_card.get_attribute("href") or ""
        if not href.startswith("http"):
            href = "https://www.yad2.co.il" + href

        first_listing_id = extract_listing_id_from_url(href)
        first_listing_url = href

        ts(f"First listing ID: {first_listing_id}")
        ts(f"First listing URL: {first_listing_url}")

        ts("Closing browser")
        await browser.close()

    ts("Finished load_first_listing")

# Run this cell, then:
await load_first_listing()

[2026-03-12 16:00:44] Starting load_first_listing with engine=chromium
[2026-03-12 16:00:44] Launching Chromium (incognito context)
[2026-03-12 16:00:46] Navigating to search URL: https://www.yad2.co.il/realestate/forsale/center-and-sharon?minPrice=1600000&maxPrice=3600000&maxFloor=4&minSquareMeterBuild=90&propertyCondition=5%2C3&page=1
[2026-03-12 16:00:49] Trying to close consent / cookie popup (if any)
[2026-03-12 16:00:54] No consent button found (timeout), continuing
[2026-03-12 16:00:54] Scrolling to load listings
[2026-03-12 16:00:56] Found 63 potential listing cards on page 1
[2026-03-12 16:00:56] First listing ID: urlhash_448627602597358454
[2026-03-12 16:00:56] First listing URL: https://www.yad2.co.il/realestate/item/center-and-sharon/o91ndxtn?opened-from=feed&component-type=main_feed&spot=platinum&location=1&pagination=1
[2026-03-12 16:00:56] Closing browser
[2026-03-12 16:00:57] Finished load_first_listing


In [9]:
from pprint import pprint
from playwright.async_api import async_playwright

# 2. Parse the first listing and inspect core fields (async version)

assert first_listing_id is not None and first_listing_url is not None, "Run the previous cell first."

listing_record = None

async def parse_first_listing():
    global listing_record

    async with async_playwright() as p:
        ts(f"Launching {BROWSER_ENGINE} for parsing")
        if BROWSER_ENGINE == "chromium":
            browser = await p.chromium.launch(headless=HEADLESS)
        else:
            browser = await p.firefox.launch(headless=HEADLESS)

        context = await browser.new_context()
        page = await context.new_page()

        ts(f"Opening listing URL: {first_listing_url}")
        await page.goto(first_listing_url, wait_until="domcontentloaded", timeout=120_000)

        # If we hit ShieldSquare Captcha on the listing page, let the user solve it
        while True:
            try:
                title = await page.title()
            except Exception:
                title = ""
            if "ShieldSquare" in title or "Captcha" in title:
                ts("Detected ShieldSquare / Captcha on listing page.")
                ts("Go to the browser window, solve the captcha / challenge until you see the real listing page.")
                input("When the full listing page is visible in the browser, press Enter here to continue...")
                await page.wait_for_timeout(2000)
                continue
            break

        # Get HTML from the async page and reuse the shared HTML parser from the pipeline
        html = await page.content()
        listing_record = scraper._extract_from_listing_html(
            html=html,
            listing_id=first_listing_id,
            original_url=first_listing_url,
            filtered_search_url=search_url_demo,
            search_page_number=1,
        )

        ts("Closing browser after parsing")
        await browser.close()

    core_view = {
        "yad2_listing_id": listing_record.yad2_listing_id,
        "original_listing_url": listing_record.original_listing_url,
        "price_ils": listing_record.price_ils,
        "rooms": listing_record.rooms,
        "built_sqm": listing_record.built_sqm,
        "floor_current": listing_record.floor_current,
        "seller_name": listing_record.seller_name,
        "seller_type": listing_record.seller_type,
        "seller_phone_normalized": listing_record.seller_phone_normalized,
        "phone_found": listing_record.phone_found,
        "phone_extraction_method": listing_record.phone_extraction_method,
        "publication_date_raw": listing_record.publication_date_raw,
        "publication_date_iso": listing_record.publication_date_iso,
        "within_last_3_months": (
            listing_record.publication_date_iso is not None
        ),
        "city": listing_record.city,
        "property_condition_label": listing_record.property_condition_label,
        "property_condition_index": listing_record.property_condition_index,
        "balcony_count": listing_record.balcony_count,
        "parking_count": listing_record.parking_count,
        "elevator": listing_record.elevator,
        "mamad": listing_record.mamad,
        "storage": listing_record.storage,
        "extra_features": listing_record.extra_features,
        "description_clean_sample": (listing_record.description_clean or "")[:280],
    }

    print("\nParsed core fields for demo listing (extended):\n")
    pprint(core_view)

# Run in the notebook
await parse_first_listing()

[2026-03-12 16:01:01] Launching chromium for parsing
[2026-03-12 16:01:02] Opening listing URL: https://www.yad2.co.il/realestate/item/center-and-sharon/o91ndxtn?opened-from=feed&component-type=main_feed&spot=platinum&location=1&pagination=1
[2026-03-12 16:01:04] Detected ShieldSquare / Captcha on listing page.
[2026-03-12 16:01:04] Go to the browser window, solve the captcha / challenge until you see the real listing page.


When the full listing page is visible in the browser, press Enter here to continue... 


[2026-03-12 16:01:56] Closing browser after parsing

Parsed core fields for demo listing (extended):

{'balcony_count': None,
 'built_sqm': None,
 'city': 'הרצליה',
 'description_clean_sample': 'בהרצליה הצעירה בקרבת רחובות קקל ואוסישקין בבנין '
                             'מעולה וחדיש דירת 5 חדרים נהדרת מרווחת עם חללים '
                             'נהדרים . יח הורים מרווחת , סלון גדול, מ.שמש '
                             'מעולה הפונה לגינה . הזדמנות לרכוש דירת 5 חדרים '
                             'במיקום מעולה חניה מקורה , מחסן פרטי , לכניסה '
                             'מיידית - מפתחות במשרד. תיווך סטאר - ארז פורת.',
 'elevator': True,
 'extra_features': None,
 'floor_current': 2,
 'mamad': True,
 'original_listing_url': 'https://www.yad2.co.il/realestate/item/center-and-sharon/o91ndxtn?opened-from=feed&component-type=main_feed&spot=platinum&location=1&pagination=1',
 'parking_count': 1,
 'phone_extraction_method': None,
 'phone_found': False,
 'price_ils': 3090000.0,
 'proper

In [5]:
# 4. (Optional) Inspect run summary from a full pipeline run

from pathlib import Path
import json

run_summary_path = Path("output/run_summary.json")

if run_summary_path.exists():
    with run_summary_path.open("r", encoding="utf-8") as f:
        summary = json.load(f)

    print("Run summary loaded from:", run_summary_path.resolve())
    for key in [
        "total_search_pages_visited",
        "total_result_cards_found",
        "total_unique_listings_found",
        "total_listings_opened",
        "total_listings_filtered_by_date",
        "total_exported_rows",
        "total_partial_rows",
        "total_rows_with_missing_critical_fields",
        "geocode_success_count",
        "geocode_failure_count",
        "route_success_count",
        "route_failure_count",
    ]:
        print(f"{key}: {summary.get(key)}")
else:
    print("No run_summary.json found yet. Run the full pipeline with `python yad2_pipeline.py` first.")


No run_summary.json found yet. Run the full pipeline with `python yad2_pipeline.py` first.


In [10]:
# 3. Inspect geocoding and driving distances (transportation / commute layer)

assert listing_record is not None, "Run the parsing cell first."

location_view = {
    "full_address_best": listing_record.full_address_best,
    "address_confidence": listing_record.address_confidence,
    "latitude": listing_record.latitude,
    "longitude": listing_record.longitude,
    "geocode_status": listing_record.geocode_status,
}

commute_view = {
    "drive_to_tel_aviv_savidor_duration_min": listing_record.drive_to_tel_aviv_savidor_duration_min,
    "drive_to_tel_aviv_savidor_distance_km": listing_record.drive_to_tel_aviv_savidor_distance_km,
    "drive_to_beer_sheva_center_duration_min": listing_record.drive_to_beer_sheva_center_duration_min,
    "drive_to_beer_sheva_center_distance_km": listing_record.drive_to_beer_sheva_center_distance_km,
    "commute_assessment": listing_record.commute_assessment,
    "likely_fit_for_tel_aviv_commuter": listing_record.likely_fit_for_tel_aviv_commuter,
    "likely_fit_for_beer_sheva_commuter": listing_record.likely_fit_for_beer_sheva_commuter,
}

print("Location / geocoding:")
pprint(location_view)

print("\nCommute and transportation distances:")
pprint(commute_view)

Location / geocoding:
{'address_confidence': 'city_only',
 'full_address_best': 'ית נדל, Israel',
 'geocode_status': 'geocode_failed',
 'latitude': None,
 'longitude': None}

Commute and transportation distances:
{'commute_assessment': None,
 'drive_to_beer_sheva_center_distance_km': None,
 'drive_to_beer_sheva_center_duration_min': None,
 'drive_to_tel_aviv_savidor_distance_km': None,
 'drive_to_tel_aviv_savidor_duration_min': None,
 'likely_fit_for_beer_sheva_commuter': False,
 'likely_fit_for_tel_aviv_commuter': False}


In [ ]:
import re
from typing import Dict, List, Optional
from bs4 import BeautifulSoup, Tag


# -----------------------------
# text helpers
# -----------------------------
def clean(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def normalize_lines_from_tag(tag: Tag) -> List[str]:
    lines = [clean(x) for x in tag.get_text("\n").split("\n")]
    return [x for x in lines if x]


# -----------------------------
# parsing helpers
# -----------------------------
BAD_DESCRIPTION_PHRASES = [
    "אין לך עדיין מודעות שאהבת",
    "כדי לשמור מודעות",
    "יש להתחבר לחשבון",
    "מודעות שאהבת",
    "מודעות נוספות",
    "שמור מודעה",
    "התחברות",
    "הרשמה",
    "מפה",
    "נווט",
    "צור קשר",
    "פרסום מודעה",
    "מודעות דומות",
    "עוד מודעות",
    "אולי יעניין אותך",
]

SECTION_STOP_MARKERS = {
    "פרטים נוספים",
    "מה יש בנכס",
    "מפה",
    "נכסים שנמכרו באזור",
    "היסטוריית שווי הנכס",
    "מודעות דומות",
}

DESCRIPTION_LABELS = [
    "תיאור הנכס",
    "על הנכס",
    "פרטים על הנכס",
    "תיאור",
]


def is_bad_ui_text(text: str) -> bool:
    t = clean(text)
    if not t:
        return True

    for phrase in BAD_DESCRIPTION_PHRASES:
        if phrase in t:
            return True

    # hex/js garbage protection
    if "_0x" in t or "function(" in t or "addEventListener" in t:
        return True

    return False


def parse_price_ils(text: str) -> Optional[int]:
    m = re.search(r"(\d{1,3}(?:,\d{3})+)\s*₪", text)
    if not m:
        m = re.search(r"(\d+)\s*₪", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_sqm(text: str) -> Optional[float]:
    m = re.search(r"(\d+(?:[.,]\d+)?)\s*מ[\"״]ר", text)
    return float(m.group(1).replace(",", ".")) if m else None


def parse_posted_date(text: str) -> Optional[str]:
    m = re.search(r"פורסם\s*ב\s*‍?(\d{2}/\d{2}/\d{2,4})", text)
    return m.group(1) if m else None


def extract_label_value_block(lines: List[str], header: str = "פרטים נוספים") -> Dict[str, str]:
    out: Dict[str, str] = {}
    try:
        start = lines.index(header) + 1
    except ValueError:
        return out

    i = start
    while i + 1 < len(lines):
        if lines[i] in SECTION_STOP_MARKERS:
            break

        label = lines[i]
        value = lines[i + 1]

        # basic sanity
        if (
            label
            and value
            and label != value
            and len(label) <= 60
            and not is_bad_ui_text(label)
            and not is_bad_ui_text(value)
        ):
            out[label] = value

        i += 2

    return out


# -----------------------------
# main-content narrowing
# -----------------------------
def find_main_listing_container(soup: BeautifulSoup) -> Tag:
    """
    Try to narrow parsing to the main property area instead of the whole page.
    """
    # Remove noisy tags first
    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    # Best anchor: find the H1 and walk upward to a large meaningful parent
    h1 = soup.find("h1")
    if h1:
        current = h1
        best = h1
        for _ in range(6):
            if not isinstance(current, Tag) or current.parent is None:
                break
            current = current.parent
            text_len = len(clean(current.get_text(" ")))
            if 300 <= text_len <= 12000:
                best = current
        return best

    # fallback: body
    return soup.body or soup


# -----------------------------
# description extraction
# -----------------------------
def description_score(text: str) -> int:
    t = clean(text)
    if not t or is_bad_ui_text(t):
        return -1000

    score = 0

    # positive real-estate signals
    positive_patterns = [
        r'מ[\"״]ר',
        r"\bבנוי\b",
        r"\bמגרש\b",
        r"\bקומה\b",
        r"\bלא\s*לתיווך\b",
        r"\bכניסה\b",
        r"\bשקט\b",
        r"\bפארק\b",
        r"\bפוטנציאל\b",
        r"\bהרחבה\b",
        r"\bהיתר(?:י)?\s*בנייה\b",
    ]
    for p in positive_patterns:
        if re.search(p, t):
            score += 3

    # punctuation often indicates natural paragraph text
    if "." in t:
        score += 2
    if "!" in t:
        score += 1
    if "," in t:
        score += 1

    # good length
    n = len(t)
    if 80 <= n <= 500:
        score += 4
    elif 50 <= n < 80:
        score += 2
    elif n > 700:
        score -= 2

    # word count
    wc = len(t.split())
    if 12 <= wc <= 80:
        score += 3
    elif wc < 8:
        score -= 3

    # penalize obvious UI-ish text
    negative_patterns = [
        r"\bהתחבר\b",
        r"\bהירשם\b",
        r"\bמודעות שאהבת\b",
        r"\bשמור מודעה\b",
        r"\bצור קשר\b",
        r"\bאולי יעניין אותך\b",
        r"\bלחשבון\b",
    ]
    for p in negative_patterns:
        if re.search(p, t):
            score -= 10

    return score


def extract_description_from_main(main_tag: Tag) -> Optional[str]:
    lines = normalize_lines_from_tag(main_tag)

    # 1) best case: explicit description label
    for i, line in enumerate(lines):
        if line in DESCRIPTION_LABELS:
            collected = []
            for j in range(i + 1, min(i + 10, len(lines))):
                nxt = clean(lines[j])
                if not nxt:
                    continue
                if nxt in SECTION_STOP_MARKERS:
                    break
                if is_bad_ui_text(nxt):
                    continue
                collected.append(nxt)

            candidate = clean(" ".join(collected))
            if candidate and description_score(candidate) > 0:
                return candidate

    # 2) otherwise score paragraph-like lines inside main container only
    candidates = []
    for line in lines:
        t = clean(line)
        if is_bad_ui_text(t):
            continue
        if len(t) < 50:
            continue
        if len(t.split()) < 8:
            continue
        candidates.append(t)

    if not candidates:
        return None

    candidates.sort(key=description_score, reverse=True)
    best = candidates[0]

    return best if description_score(best) > 0 else None


# -----------------------------
# complete parser
# -----------------------------
def parse_yad2_html_reliable(html: str, url: str = "") -> Dict[str, Optional[str]]:
    soup = BeautifulSoup(html, "html.parser")

    # detect block page before narrowing
    raw_text = clean(soup.get_text(" "))
    if "are you for real" in raw_text.lower() or "hcaptcha" in raw_text.lower():
        return {
            "url": url,
            "price": None,
            "address": None,
            "floor": None,
            "sqm_built": None,
            "description": None,
            "published": None,
            "error": "Blocked page / captcha detected",
        }

    main_tag = find_main_listing_container(soup)
    lines = normalize_lines_from_tag(main_tag)
    page_text = " ".join(lines)

    h1 = soup.find("h1")
    address = clean(h1.get_text(" ", strip=True)) if h1 else None

    details = extract_label_value_block(lines, "פרטים נוספים")

    price = parse_price_ils(page_text)
    published = parse_posted_date(page_text)
    description = extract_description_from_main(main_tag)

    floor = None
    if "קרקע" in lines:
        floor = 0
    elif "קומה" in details:
        val = details["קומה"]
        if "קרקע" in val:
            floor = 0
        else:
            m = re.search(r"(\d+)", val)
            floor = int(m.group(1)) if m else None

    sqm_built = None
    sqm_raw = details.get("מ״ר בנוי") or details.get('מ"ר בנוי')
    if sqm_raw:
        sqm_built = parse_sqm(sqm_raw)

    return {
        "url": url,
        "price": price,
        "address": address,
        "floor": floor,
        "sqm_built": sqm_built,
        "description": description,
        "published": published,
        "error": None,
    }

In [ ]:
from playwright.async_api import async_playwright

async def fetch_and_parse(url: str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(2500)

        html = await page.content()
        await browser.close()

    return parse_yad2_html_reliable(html, url)

In [ ]:
row = await fetch_and_parse("https://www.yad2.co.il/realestate/item/center-and-sharon/cvcoi0hh")
print(row["description"])

In [ ]:
row

In [ ]:
import asyncio
import csv
import os
import re
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


# =========================================================
# CONFIG
# =========================================================

BASE_DIR = Path("yad2_export")
IMAGES_DIR = BASE_DIR / "images"
XLSX_PATH = BASE_DIR / "products.xlsx"

BASE_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
}

# Same-site category support: just replace this URL with any other Yad2 market category URL.
CATEGORY_URL = "https://www.yad2.co.il/market/collections/sports-and-leisure_riding-tool_bicycle"

# Client-side filters after scraping cards/details.
FILTERS = {
    "min_price": None,       # e.g. 500
    "max_price": None,       # e.g. 2500
    "location_contains": None,  # e.g. "ירושלים"
}

MAX_PAGES = 3            # raise as needed
MAX_ITEMS = 40           # total items cap
HEADLESS = False         # False is more reliable on anti-bot sites


# =========================================================
# HELPERS
# =========================================================

def clean(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).strip()

def slugify(text: str, max_len: int = 80) -> str:
    text = clean(text)
    text = re.sub(r"[^\w\u0590-\u05FF\-\. ]+", "", text, flags=re.UNICODE)
    text = text.replace(" ", "_")
    return text[:max_len].strip("_") or "item"

def parse_price(text: str) -> Optional[int]:
    if not text:
        return None
    m = re.search(r"(\d{1,3}(?:,\d{3})+|\d+)\s*₪", text)
    if not m:
        return None
    return int(m.group(1).replace(",", ""))

def parse_phone(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.search(r"(0\d{1,2}[- ]?\d{3,4}[- ]?\d{4})", text)
    return m.group(1) if m else None

def looks_blocked(text: str) -> bool:
    t = (text or "").lower()
    return (
        "are you for real" in t
        or "hcaptcha" in t
        or "cf_input" in t
    )

def passes_filters(row: Dict[str, Any], filters: Dict[str, Any]) -> bool:
    price = row.get("price")
    location = row.get("seller_location") or row.get("card_location") or ""

    if filters.get("min_price") is not None and (price is None or price < filters["min_price"]):
        return False
    if filters.get("max_price") is not None and (price is None or price > filters["max_price"]):
        return False
    if filters.get("location_contains"):
        if filters["location_contains"] not in location:
            return False
    return True

def extract_json_ld(soup: BeautifulSoup) -> List[dict]:
    objs = []
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        txt = tag.string or tag.get_text()
        txt = txt.strip()
        if not txt:
            continue
        try:
            parsed = json.loads(txt)
            if isinstance(parsed, list):
                objs.extend(x for x in parsed if isinstance(x, dict))
            elif isinstance(parsed, dict):
                objs.append(parsed)
        except Exception:
            pass
    return objs


# =========================================================
# DATA MODEL
# =========================================================

@dataclass
class ProductRow:
    listing_id: str
    title: Optional[str]
    price: Optional[int]
    seller_name: Optional[str]
    seller_location: Optional[str]
    seller_other_products_count: Optional[int]
    description: Optional[str]
    category: Optional[str]
    subcategory: Optional[str]
    image_urls: Optional[str]
    downloaded_images: Optional[str]
    original_link: Optional[str]
    phone_number: Optional[str]
    card_location: Optional[str]
    card_badges: Optional[str]
    error: Optional[str]


# =========================================================
# LIST PAGE EXTRACTION
# =========================================================

async def extract_card_links(page) -> List[dict]:
    """
    Heuristic extraction from category pages.
    Works across Yad2 market categories better than brittle CSS-only logic.
    """
    cards = await page.locator("a").evaluate_all("""
    (anchors) => anchors.map(a => {
      const href = a.href || "";
      const txt = (a.innerText || "").trim();
      const img = a.querySelector("img");
      return {
        href,
        text: txt,
        img: img ? (img.src || img.getAttribute("src") || "") : ""
      };
    })
    """)
    results = []
    seen = set()

    for a in cards:
        href = a.get("href", "")
        text = clean(a.get("text", ""))
        img = a.get("img", "")

        # Heuristic: Yad2 item pages are product-detail URLs under /market/
        # Adjust here if Yad2 changes URL shape.
        if "/market/" not in href:
            continue
        if "/collections/" in href:
            continue
        if href in seen:
            continue

        # Must look like a result card, not just nav
        if len(text) < 3 and not img:
            continue

        results.append({
            "href": href,
            "card_text": text,
            "card_img": img,
        })
        seen.add(href)

    return results


def parse_card_preview(card_text: str) -> Dict[str, Any]:
    lines = [clean(x) for x in card_text.splitlines() if clean(x)]
    joined = " | ".join(lines)

    price = None
    location = None
    title = None
    badges = []

    for i, ln in enumerate(lines):
        if price is None:
            p = parse_price(ln)
            if p is not None:
                price = p
                continue

    # Heuristic from current category page structure:
    # price line(s), then location, then title, then condition/badges. :contentReference[oaicite:1]{index=1}
    if lines:
        # find first non-price line after a price as location, next as title
        price_idx = None
        for i, ln in enumerate(lines):
            if parse_price(ln) is not None:
                price_idx = i
                break
        if price_idx is not None:
            after = lines[price_idx + 1:]
            non_price_after = [x for x in after if parse_price(x) is None]
            if len(non_price_after) >= 1:
                location = non_price_after[0]
            if len(non_price_after) >= 2:
                title = non_price_after[1]
            if len(non_price_after) > 2:
                badges = non_price_after[2:]

    return {
        "card_price": price,
        "card_location": location,
        "card_title": title,
        "card_badges": ", ".join(badges) if badges else None,
        "card_text_joined": joined,
    }


# =========================================================
# DETAIL PAGE EXTRACTION
# =========================================================

async def get_html(page, url: str) -> str:
    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_timeout(2200)
    html = await page.content()
    if looks_blocked(html):
        print(f"Blocked/captcha at: {url}")
        input("Solve the challenge in the browser, then press Enter here...")
        await page.wait_for_timeout(1500)
        html = await page.content()
    return html

def parse_breadcrumbs(soup: BeautifulSoup) -> Dict[str, Optional[str]]:
    crumbs = []
    for tag in soup.find_all(["a", "li", "span"]):
        txt = clean(tag.get_text(" "))
        if txt and len(txt) < 80:
            crumbs.append(txt)

    # light heuristic (supports baby, bikes/sport, and other market categories)
    category = None
    subcategory = None
    if "ילדים ותינוקות" in crumbs:
        category = "ילדים ותינוקות"
    if "עגלות וטיולונים" in crumbs:
        subcategory = "עגלות וטיולונים"
    if "ספורט ופנאי" in crumbs:
        category = "ספורט ופנאי"
    if "אופניים" in crumbs or "כלי רכיבה" in crumbs:
        subcategory = "אופניים"
    return {"category": category, "subcategory": subcategory}

def parse_seller_count(text: str) -> Optional[int]:
    patterns = [
        r"(\d+)\s+מודעות\s+נוספות",
        r"(\d+)\s+מוצרים\s+נוספים",
        r"למוכר\s+זה\s+(\d+)\s+מודעות",
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            return int(m.group(1))
    return None

def parse_detail_html(html: str, url: str, preview: Dict[str, Any]) -> Dict[str, Any]:
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style", "noscript", "svg"]):
        t.decompose()

    full_text = clean(soup.get_text("\n"))
    lines = [clean(x) for x in soup.get_text("\n").split("\n") if clean(x)]

    title = None
    h1 = soup.find("h1")
    if h1:
        title = clean(h1.get_text(" "))

    price = parse_price(full_text) or preview.get("card_price")

    # seller name / location heuristics
    seller_name = None
    seller_location = None
    phone_number = parse_phone(full_text)
    seller_other_products_count = parse_seller_count(full_text)

    # try JSON-LD first
    for obj in extract_json_ld(soup):
        if not title and isinstance(obj.get("name"), str):
            title = clean(obj["name"])
        if not seller_name:
            seller = obj.get("seller") or obj.get("author")
            if isinstance(seller, dict):
                seller_name = clean(seller.get("name", "")) or seller_name

    # line-based fallback
    for i, ln in enumerate(lines):
        if seller_name is None and re.search(r"(המוכר|מוכר|פרטי|עסק)", ln):
            if i + 1 < len(lines):
                nxt = lines[i + 1]
                if len(nxt) < 80:
                    seller_name = nxt

        if seller_location is None and re.search(r"(אזור|מיקום|עיר|יישוב)", ln):
            if i + 1 < len(lines):
                nxt = lines[i + 1]
                if len(nxt) < 80:
                    seller_location = nxt

    if not seller_location:
        seller_location = preview.get("card_location")

    # description candidate
    bad_desc = [
        "אין לך עדיין מודעות שאהבת",
        "כדי לשמור מודעות",
        "יש להתחבר לחשבון",
        "מודעות שאהבתי",
        "אולי יעניין אותך",
    ]
    desc_candidates = []
    for ln in lines:
        if len(ln) < 40:
            continue
        if any(b in ln for b in bad_desc):
            continue
        desc_candidates.append(ln)

    description = None
    if desc_candidates:
        def score(s: str) -> int:
            sc = 0
            if "." in s: sc += 2
            if "," in s: sc += 1
            if len(s) >= 80: sc += 2
            if "מ״ר" in s or 'מ"ר' in s: sc += 1
            if "חדש" in s or "כמו חדש" in s or "במצב" in s: sc += 1
            return sc
        desc_candidates.sort(key=score, reverse=True)
        description = desc_candidates[0]

    bc = parse_breadcrumbs(soup)

    # images from HTML after render
    image_urls = []
    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or ""
        src = src.strip()
        if not src:
            continue
        if src.startswith("data:"):
            continue
        if "yad2" in src or src.startswith("http"):
            image_urls.append(src)

    # de-dup and keep likely product images, not logos/icons
    filtered = []
    seen = set()
    for u in image_urls:
        lu = u.lower()
        if any(x in lu for x in ["logo", "icon", "avatar", "sprite"]):
            continue
        if u not in seen:
            filtered.append(u)
            seen.add(u)

    listing_id = slugify(url.rstrip("/").split("/")[-1] or title or "item", 40)

    return {
        "listing_id": listing_id,
        "title": title or preview.get("card_title"),
        "price": price,
        "seller_name": seller_name,
        "seller_location": seller_location,
        "seller_other_products_count": seller_other_products_count,
        "description": description,
        "category": bc["category"],
        "subcategory": bc["subcategory"],
        "image_urls": filtered,
        "original_link": url,
        "phone_number": phone_number,
        "card_location": preview.get("card_location"),
        "card_badges": preview.get("card_badges"),
        "error": None,
    }


# =========================================================
# IMAGE DOWNLOAD
# =========================================================

def download_images(listing_id: str, title: Optional[str], image_urls: List[str]) -> List[str]:
    saved = []
    title_slug = slugify(title or listing_id, 60)

    sess = requests.Session()
    sess.headers.update(HEADERS)

    for idx, url in enumerate(image_urls[:12], start=1):
        try:
            ext = os.path.splitext(urlparse(url).path)[1].lower()
            if ext not in [".jpg", ".jpeg", ".png", ".webp"]:
                ext = ".jpg"

            fname = f"{listing_id}__{title_slug}__img{idx:02d}{ext}"
            fpath = IMAGES_DIR / fname

            r = sess.get(url, timeout=30)
            r.raise_for_status()
            with open(fpath, "wb") as f:
                f.write(r.content)
            saved.append(str(fpath.relative_to(BASE_DIR)))
        except Exception:
            continue

    return saved


# =========================================================
# MAIN PIPELINE
# =========================================================

async def scrape_category(category_url: str,
                          max_pages: int = 3,
                          max_items: int = 40,
                          filters: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    filters = filters or {}
    rows: List[Dict[str, Any]] = []
    seen_urls = set()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        context = await browser.new_context(
            locale="he-IL",
            viewport={"width": 1440, "height": 1200},
            user_agent=HEADERS["User-Agent"],
        )
        list_page = await context.new_page()
        detail_page = await context.new_page()

        for page_num in range(1, max_pages + 1):
            url = category_url if page_num == 1 else f"{category_url}?pageNumber={page_num}"
            print(f"Scanning list page {page_num}: {url}")
            html = await get_html(list_page, url)

            if looks_blocked(html):
                print("Still blocked on category page; stopping.")
                break

            card_links = await extract_card_links(list_page)

            # fallback: if no anchors extracted, stop
            if not card_links:
                print("No listing links found on this page.")
                break

            for card in card_links:
                item_url = card["href"]
                if item_url in seen_urls:
                    continue
                seen_urls.add(item_url)

                preview = parse_card_preview(card.get("card_text", ""))

                try:
                    item_html = await get_html(detail_page, item_url)
                    item = parse_detail_html(item_html, item_url, preview)

                    if not passes_filters(item, filters):
                        continue

                    saved_images = download_images(
                        listing_id=item["listing_id"],
                        title=item.get("title"),
                        image_urls=item.get("image_urls") or [],
                    )
                    item["downloaded_images"] = "; ".join(saved_images) if saved_images else None
                    item["image_urls"] = "; ".join(item["image_urls"]) if item.get("image_urls") else None

                    rows.append(item)
                    print(f"  ✓ {item.get('title')}")
                except Exception as e:
                    rows.append(asdict(ProductRow(
                        listing_id=slugify(item_url.split("/")[-1], 40),
                        title=preview.get("card_title"),
                        price=preview.get("card_price"),
                        seller_name=None,
                        seller_location=preview.get("card_location"),
                        seller_other_products_count=None,
                        description=None,
                        category=None,
                        subcategory=None,
                        image_urls=None,
                        downloaded_images=None,
                        original_link=item_url,
                        phone_number=None,
                        card_location=preview.get("card_location"),
                        card_badges=preview.get("card_badges"),
                        error=str(e),
                    )))

                if len(rows) >= max_items:
                    break

            if len(rows) >= max_items:
                break

        await context.close()
        await browser.close()

    return rows


def export_xlsx(rows: List[Dict[str, Any]], path: Path = XLSX_PATH) -> None:
    df = pd.DataFrame(rows)

    columns = [
        "listing_id",
        "title",
        "price",
        "seller_name",
        "seller_location",
        "seller_other_products_count",
        "description",
        "category",
        "subcategory",
        "original_link",
        "phone_number",
        "card_location",
        "card_badges",
        "image_urls",
        "downloaded_images",
        "error",
    ]
    for c in columns:
        if c not in df.columns:
            df[c] = None
    df = df[columns]

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="products")

    print(f"Saved: {path}")


async def main():
    rows = await scrape_category(
        category_url=CATEGORY_URL,
        max_pages=MAX_PAGES,
        max_items=MAX_ITEMS,
        filters=FILTERS,
    )
    export_xlsx(rows)
    print(f"Saved images in: {IMAGES_DIR}")


# In a normal .py file:
# if __name__ == "__main__":
#     asyncio.run(main())

# In Jupyter:
await main()

In [11]:
import requests
import json

url = "https://gw.yad2.co.il/realestate-feed/locations"

r = requests.get(url)
print(r)
data = r.json()

result = {"cities": {}}

for city in data["cities"]:
    city_id = city["id"]
    result["cities"][city_id] = {
        "name": city["text"],
        "neighborhoods": {}
    }

    for n in city.get("neighborhoods", []):
        result["cities"][city_id]["neighborhoods"][n["id"]] = n["text"]

with open("yad2_locations.json", "w", encoding="utf8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

<Response [404]>


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [4]:
import pandas as pd
import os
import chardet

def fix_hebrew_encoding(text):
    if not isinstance(text, str):
        return text
    # Try to repair "mojibake" (Hebrew looking like latin-1)
    try:
        # Most common Yad2/Excel mess: latin-1 -> windows-1255
        return text.encode('latin-1').decode('windows-1255')
    except:
        try:
            # Alternative: if it was already UTF-8 but read as latin-1
            return text.encode('latin-1').decode('utf-8')
        except:
            return text

def process_file_decode(input_path, output_path):
    file_ext = os.path.splitext(input_path)[1].lower()
   
    # --- STEP 1: SMART READ ---
    if file_ext == '.csv':
        # Detect the encoding of the CSV file automatically
        with open(input_path, 'rb') as f:
            result = chardet.detect(f.read(10000))
            detected_enc = result['encoding']
       
        print(f"Detected CSV encoding: {detected_enc}")
        # Read using the detected encoding
        df = pd.read_csv(input_path, encoding=detected_enc)
    else:
        # Read Excel (.xlsx or .xls)
        df = pd.read_excel(input_path)

    # --- STEP 2: APPLY THE REPAIR ---
    # We apply the fix to every string cell
    df_fixed = df.map(fix_hebrew_encoding)

    # --- STEP 3: SMART WRITE ---
    if output_path.lower().endswith('.xlsx'):
        # Saving as a real Excel file
        df_fixed.to_excel(output_path, index=False, engine='openpyxl')
    else:
        # Saving as CSV with BOM (utf-8-sig) so Excel opens it in Hebrew
        df_fixed.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"✅ Success! Fixed file saved to: {output_path}")



input_path = "/Users/nircko/Downloads/yad2_output2/yad2_output2/listings_full.csv"

output_path = "/Users/nircko/Downloads/yad2_output2/yad2_output2/fixed_hebrew_file.xlsx"

process_file_decode(input_path,output_path)



Detected CSV encoding: utf-8
✅ Success! Fixed file saved to: /Users/nircko/Downloads/yad2_output2/yad2_output2/fixed_hebrew_file.xlsx
